In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
books = pd.read_csv('/home/ananta/personnel_study_material/practice-and-projects/Sample_data/books/data.csv')

In [3]:
books.columns

Index(['isbn13', 'isbn10', 'title', 'subtitle', 'authors', 'categories',
       'thumbnail', 'description', 'published_year', 'average_rating',
       'num_pages', 'ratings_count'],
      dtype='object')

In [4]:
books['combined_text'] = (
    books['title'].fillna('') + ' ' +
    books['subtitle'].fillna('') + ' ' +
    books['authors'].fillna('') + ' ' +
    books['categories'].fillna('') + ' ' +
    books['description'].fillna('')
)

In [5]:
cv = CountVectorizer(stop_words='english')
cv_matrix = cv.fit_transform(books['combined_text'])

In [6]:
similarity = cosine_similarity(cv_matrix)

In [7]:
def recommend_book(title, n=5):
    # if title not exact, try match
    if title not in books['title'].values:
        print("Title not found, trying partial match...")
        title_match = books[books['title'].str.contains(title, case=False, na=False)]
        if len(title_match) == 0:
            return "No similar title found"
        title = title_match.iloc[0]['title']
        print(f"Using closest match: {title}")
    
    index = books[books['title']==title].index[0]
    scores = list(enumerate(similarity[index]))
    sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    
    book_indices = [i[0] for i in sorted_scores]
    return books[['title','authors','categories']].iloc[book_indices]

In [8]:
recommend_book("Harry Potter")

,title,authors,categories
2730,The Harry Potter Collection,J. K. Rowling,Juvenile Fiction
2710,Harry Potter and the Prisoner of Azkaban (Book 3),"Rowling, J.K.",Juvenile Fiction
2697,Harry Potter and the Chamber of Secrets,J. K. Rowling;Mary GrandPre,Juvenile Fiction
2661,Harry Potter and the Chamber of Secrets (Book 2),"Rowling, J.K.",Juvenile Fiction
2698,Harry Potter and the Sorcerer's Stone (Book 1),"Rowling, J.K.",Juvenile Fiction
